# Data Fusion Track 2 — 0.86+ push (Kaggle-adapted)

Основа взята из `d-ulybin/data_fusion_track_2` (public 0.8536), дополнена более агрессивным суперблендингом:
- per-target rank blend с более мелкой сеткой;
- meta-combo: blend + stacking + (optional) lgbm_meta;
- holdout-оптимизация весов по target-ам;
- финальный super submission `submissions/superblend.parquet`.


In [1]:

# 1) Setup: clone solution repo + install deps + patch paths for Kaggle
import os, re, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/d-ulybin/data_fusion_track_2"
REPO_DIR = Path("/kaggle/working/data_fusion_track_2")
DATA_DIR = "/kaggle/input/datasets/hatab123/data-fusion-contest-2026/"  # <-- Kaggle input path

if REPO_DIR.exists():
    subprocess.run(["rm", "-rf", str(REPO_DIR)], check=False)

subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=False)

# patch DATA_DIR and some blend granularity for stronger search
utils_path = REPO_DIR / "utils.py"
s = utils_path.read_text(encoding="utf-8")
s = re.sub(r'DATA_DIR\s*=\s*"[^"]+"', f'DATA_DIR = "{DATA_DIR}"', s)
utils_path.write_text(s, encoding="utf-8")

# finer search in blending/stacking scripts
for rel in ["06_blend.py", "07_stacking.py"]:
    p = REPO_DIR / rel
    t = p.read_text(encoding="utf-8")
    t = t.replace("step=0.10", "step=0.05")
    t = t.replace("np.arange(0, 1.05, 0.05)", "np.arange(0, 1.02, 0.02)")
    p.write_text(t, encoding="utf-8")

print("Repo ready:", REPO_DIR)
print("DATA_DIR set to:", DATA_DIR)


Cloning into '/kaggle/working/data_fusion_track_2'...


Repo ready: /kaggle/working/data_fusion_track_2
DATA_DIR set to: /kaggle/input/datasets/hatab123/data-fusion-contest-2026/


In [2]:
# 2) Resume-aware pipeline (uses checkpoints if found; otherwise runs full training)
# Fix:
# - 06_blend.py patched: if checkpoints_nn/fold_0.npz missing -> load NN from blend_artifacts/blend_data.npz
# - If checkpoints are present -> skip retraining corresponding steps
# - If something is missing -> run usual steps 01..05 as needed
# - ETA for steps 05/06/07: one line every ETA_INTERVAL_SEC (default 120s)

import os, re, subprocess, sys, shutil, time, textwrap
from pathlib import Path

REPO_DIR = Path("/kaggle/working/data_fusion_track_2")
os.chdir(REPO_DIR)

LOG_DIR = REPO_DIR / "logs_parallel"
LOG_DIR.mkdir(exist_ok=True)

# =========================
# YOUR CHECKPOINTS LOCATION
# =========================
CKPT_SRC_DIR = Path("/kaggle/input/datasets/nohimilk/checpoints")

SRC_CANDIDATES = {
    "blend": [
        CKPT_SRC_DIR / "blend_data.npz",
    ],
    "lgbm": [
        CKPT_SRC_DIR / "lgbm_predictions (2).npz",
        CKPT_SRC_DIR / "lgbm_predictions.npz",
    ],
    "lgbm_meta": [
        CKPT_SRC_DIR / "lgbm_predictions _meta(2).npz",
        CKPT_SRC_DIR / "lgbm_predictions_meta(2).npz",
        CKPT_SRC_DIR / "lgbm_predictions_meta.npz",
    ],
    "pyboost": [
        CKPT_SRC_DIR / "pyboost_predictions.npz",
    ],
}

DST = {
    "blend": REPO_DIR / "blend_artifacts" / "blend_data.npz",
    "lgbm": REPO_DIR / "checkpoints_lgbm" / "lgbm_predictions.npz",
    "lgbm_meta": REPO_DIR / "checkpoints_lgbm_meta" / "lgbm_predictions.npz",
    "pyboost": REPO_DIR / "checkpoints_pyboost" / "pyboost_predictions.npz",
}

# ===== настройки =====
FORCE_RESTORE = True     # overwrite local checkpoints with input ones, if found
FORCE_RERUN_06 = True    # rerun blend even if exists
FORCE_RERUN_07 = True    # rerun stacking even if exists
FORCE_RERUN_05 = False   # normally False because you have meta checkpoint

# If you really want retrain even when checkpoint exists (usually keep False)
FORCE_RETRAIN_01 = False
FORCE_RETRAIN_02 = False
FORCE_RETRAIN_03 = False
FORCE_RETRAIN_04 = False
FORCE_RETRAIN_05 = False

ETA_INTERVAL_SEC = 120   # print 1 ETA line every 2 minutes
PARALLEL_POLL_SEC = 300  # parallel status every 5 minutes

# ---------------------------
# helpers
# ---------------------------
def run(cmd, check=False):
    return subprocess.run(["bash", "-lc", cmd], check=check)

def exists(p: Path) -> bool:
    return p.exists()

def safe_mkdir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def safe_unlink(p: Path):
    if p.exists():
        p.unlink()

def safe_backup(p: Path):
    if p.exists():
        bak = p.with_suffix(p.suffix + ".bak")
        if not bak.exists():
            shutil.copy2(p, bak)

def find_first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None

def copy_checkpoint(src: Path, dst: Path, force: bool):
    safe_mkdir(dst.parent)
    if dst.exists() and not force:
        return False
    shutil.copy2(src, dst)
    return True

def run_parallel_no_limits_quiet(scripts, poll_sec=PARALLEL_POLL_SEC):
    procs = {}
    for s in scripts:
        log_path = LOG_DIR / f"{s}.log"
        f = open(log_path, "w", buffering=1)
        p = subprocess.Popen([sys.executable, s], cwd=str(REPO_DIR), stdout=f, stderr=subprocess.STDOUT)
        procs[s] = (p, f, log_path)
        print(f"[PARALLEL] started {s} -> {log_path.name}")

    while True:
        alive = [s for s,(p,_,_) in procs.items() if p.poll() is None]
        if not alive:
            break
        print(f"[PARALLEL] running: {alive}")
        time.sleep(poll_sec)

    failed = []
    for s,(p,f,lp) in procs.items():
        rc = p.poll()
        f.close()
        print(f"[PARALLEL] done {s} rc={rc} (log {lp.name})")
        if rc != 0:
            failed.append((s, rc, lp))

    if failed:
        msg = []
        for s, rc, lp in failed:
            try:
                tail = subprocess.check_output(["bash","-lc", f"tail -n 160 {lp}"], text=True)
            except Exception:
                tail = "(tail failed)"
            msg.append(f"\n{s} rc={rc}\n--- tail {lp.name} ---\n{tail}")
        raise RuntimeError("Some parallel jobs failed:" + "".join(msg))

def run_logged_with_eta(script: str, typical_sec: int, progress_re=None, interval_sec: int = ETA_INTERVAL_SEC):
    log_path = LOG_DIR / f"{script}.log"
    safe_mkdir(log_path.parent)
    start = time.time()

    f = open(log_path, "w", buffering=1)
    p = subprocess.Popen([sys.executable, script], cwd=str(REPO_DIR), stdout=f, stderr=subprocess.STDOUT)

    def fmt(sec):
        sec = max(0, int(sec))
        h = sec // 3600
        m = (sec % 3600) // 60
        s = sec % 60
        return f"{h:02d}:{m:02d}:{s:02d}" if h else f"{m:02d}:{s:02d}"

    last_done, last_total = None, None

    while p.poll() is None:
        time.sleep(interval_sec)
        elapsed = time.time() - start

        try:
            tail_txt = subprocess.check_output(["bash","-lc", f"tail -n 200 {log_path}"], text=True)
        except Exception:
            tail_txt = ""

        last_line = ""
        for line in reversed(tail_txt.splitlines()):
            if line.strip():
                last_line = line.strip()
                break

        if progress_re:
            ms = list(re.finditer(progress_re, tail_txt))
            if ms:
                m = ms[-1]
                try:
                    last_done = int(m.group(1))
                    last_total = int(m.group(2))
                except Exception:
                    pass

        if last_done is not None and last_total is not None and last_done > 0:
            eta = (elapsed / last_done) * (last_total - last_done)
        else:
            eta = max(0, typical_sec - elapsed)

        print(f"[{script}] elapsed {fmt(elapsed)} | ETA ~ {fmt(eta)} | last: {last_line}")

    rc = p.poll()
    f.close()

    total_elapsed = time.time() - start
    if rc != 0:
        try:
            tail_err = subprocess.check_output(["bash","-lc", f"tail -n 160 {log_path}"], text=True)
        except Exception:
            tail_err = "(tail failed)"
        raise RuntimeError(f"{script} failed rc={rc}\n--- tail {log_path.name} ---\n{tail_err}")

    print(f"[{script}] DONE in {fmt(total_elapsed)} (log: {log_path.name})")

# ---------------------------
# Patch 06_blend.py to not require NN fold checkpoints (fallback to blend_data.npz)
# ---------------------------
def patch_06_blend_nn_fallback():
    p = REPO_DIR / "06_blend.py"
    if not p.exists():
        print("[WARN] 06_blend.py not found, skip patch")
        return

    t0 = p.read_text(encoding="utf-8")
    # already patched?
    if "blend_artifacts/blend_data.npz" in t0 and "FALLBACK_NN_FROM_BLEND_DATA" in t0:
        return

    safe_backup(p)

    # Insert fallback right after def load_nn():
    pat = r"(def\s+load_nn\s*\(\s*\)\s*:\s*\n)"
    ins = r"""\1    # FALLBACK_NN_FROM_BLEND_DATA
    # If NN fold checkpoints are missing, use blend_artifacts/blend_data.npz (oof_nn/test_nn).
    import os
    import numpy as np
    nn_fold0 = os.path.join("checkpoints_nn", "fold_0.npz")
    if not os.path.exists(nn_fold0):
        bd = os.path.join("blend_artifacts", "blend_data.npz")
        if os.path.exists(bd):
            d = np.load(bd)
            if ("oof_nn" in d) and ("test_nn" in d):
                return d["oof_nn"], d["test_nn"]
        raise FileNotFoundError(f"NN checkpoints not found at {nn_fold0} and no fallback at {bd}")
"""
    t, n = re.subn(pat, ins, t0, count=1, flags=re.M)
    if n == 0:
        # if structure differs, do nothing
        print("[WARN] Could not patch load_nn() in 06_blend.py (pattern not found).")
        return

    p.write_text(t, encoding="utf-8")
    print("[PATCH] 06_blend.py patched to use blend_data.npz when NN folds missing")

patch_06_blend_nn_fallback()

# ---------------------------
# 0) Restore checkpoints if present
# ---------------------------
restored = {}
for key, candidates in SRC_CANDIDATES.items():
    src = find_first_existing(candidates)
    if src is None:
        restored[key] = False
        continue
    restored[key] = copy_checkpoint(src, DST[key], force=FORCE_RESTORE)

print("[RESTORE]", restored)

# ---------------------------
# 1) Build submissions from restored checkpoints when possible
# ---------------------------
sys.path.insert(0, str(REPO_DIR))
from utils import DATA_DIR as DATA_DIR

import numpy as np
import polars as pl

SUB_DIR = REPO_DIR / "submissions"
safe_mkdir(SUB_DIR)

sample = pl.read_parquet(str(Path(DATA_DIR) / "sample_submit.parquet"))
predict_cols = [c for c in sample.columns if c.startswith("predict_")]
cid = sample["customer_id"]

def npz_get_test_preds(npz_path: Path):
    d = np.load(npz_path)
    for k in ["test_preds", "test_pred", "test"]:
        if k in d:
            return d[k]
    raise KeyError(f"{npz_path} has no test preds keys. keys={list(d.keys())}")

def write_submit(preds: np.ndarray, out_path: Path):
    preds = np.asarray(preds)
    if preds.ndim != 2:
        raise ValueError(f"{out_path}: preds must be 2D, got {preds.shape}")
    if preds.shape[1] != len(predict_cols):
        raise ValueError(f"{out_path}: preds cols={preds.shape[1]} but need {len(predict_cols)}")
    sub = pl.DataFrame({"customer_id": cid}).hstack(
        pl.DataFrame(preds.astype(np.float64), schema=predict_cols)
    )
    sub.write_parquet(str(out_path))

# lgbm / pyboost / lgbm_meta
if DST["lgbm"].exists():
    write_submit(npz_get_test_preds(DST["lgbm"]), SUB_DIR / "lgbm.parquet")
if DST["pyboost"].exists():
    write_submit(npz_get_test_preds(DST["pyboost"]), SUB_DIR / "pyboost.parquet")
if DST["lgbm_meta"].exists():
    write_submit(npz_get_test_preds(DST["lgbm_meta"]), SUB_DIR / "lgbm_meta.parquet")

# nn.parquet from blend_data.npz
if DST["blend"].exists():
    bd = np.load(DST["blend"])
    if "test_nn" in bd:
        write_submit(bd["test_nn"], SUB_DIR / "nn.parquet")

# Optionally rerun 06/07
if FORCE_RERUN_06:
    safe_unlink(SUB_DIR / "blend.parquet")
if FORCE_RERUN_07:
    safe_unlink(SUB_DIR / "stacking.parquet")

# ---------------------------
# 2) Determine what exists
# ---------------------------
HAVE_01 = exists(REPO_DIR / "features" / "train_features.parquet") and exists(REPO_DIR / "features" / "meta.json")
HAVE_NN_FOLDS = exists(REPO_DIR / "checkpoints_nn" / "fold_0.npz")  # only if you trained 02
HAVE_NN_SUB = exists(SUB_DIR / "nn.parquet")                        # enough for 06/07
HAVE_03 = exists(DST["lgbm"])
HAVE_04 = exists(DST["pyboost"])
HAVE_05 = exists(DST["lgbm_meta"])
HAVE_06 = exists(SUB_DIR / "blend.parquet")
HAVE_07 = exists(SUB_DIR / "stacking.parquet")

print(f"[STATE] features={HAVE_01} | nn_sub={HAVE_NN_SUB} | nn_folds={HAVE_NN_FOLDS} | "
      f"lgbm={HAVE_03} | pyboost={HAVE_04} | meta={HAVE_05} | blend={HAVE_06} | stacking={HAVE_07}")

# ---------------------------
# 3) Decide training needs (fallback to "usual" if missing)
# ---------------------------
need_01 = FORCE_RETRAIN_01 or (not HAVE_01)
need_03 = FORCE_RETRAIN_03 or (not HAVE_03)
need_04 = FORCE_RETRAIN_04 or (not HAVE_04)
need_05 = FORCE_RETRAIN_05 or (not HAVE_05)

# For step05 training we MUST have NN folds, not just nn.parquet
need_02_for_05 = need_05 and (not HAVE_NN_FOLDS)
need_02_for_06_07 = (not HAVE_NN_SUB) and (not DST["blend"].exists())  # if no NN submission and no blend_data fallback
need_02 = FORCE_RETRAIN_02 or need_02_for_05 or need_02_for_06_07

# ---------------------------
# 4) Run training steps if needed
# ---------------------------
# Step01
if need_01:
    print("[RUN] 01_feature_engineering.py")
    run_logged_with_eta("01_feature_engineering.py", typical_sec=10*60, interval_sec=ETA_INTERVAL_SEC)
    HAVE_01 = True
else:
    print("[SKIP] 01")

# Steps 02 and 04 can be parallel (no limits)
par_jobs = []
if need_02:
    par_jobs.append("02_train_nn.py")
if need_04:
    par_jobs.append("04_train_pyboost.py")

if par_jobs:
    if not HAVE_01 and (need_02 or need_04 or need_03 or need_05):
        raise RuntimeError("Features missing: step01 is required before training.")
    print("[RUN] parallel:", par_jobs)
    run_parallel_no_limits_quiet(par_jobs)
    HAVE_NN_FOLDS = exists(REPO_DIR / "checkpoints_nn" / "fold_0.npz")
    HAVE_NN_SUB = exists(SUB_DIR / "nn.parquet")  # if script writes it
else:
    print("[SKIP] 02/04")

# Step03 (only if lgbm checkpoint missing)
if need_03:
    if not HAVE_01:
        raise RuntimeError("Need features for training 03 but step01 features missing.")
    print("[RUN] 03_train_lgbm.py")
    run_logged_with_eta("03_train_lgbm.py", typical_sec=6*3600, interval_sec=ETA_INTERVAL_SEC)
    HAVE_03 = exists(DST["lgbm"])
else:
    print("[SKIP] 03 (using checkpoint)")

# Step05 (only if meta checkpoint missing)
if need_05:
    if not (HAVE_01 and HAVE_03 and HAVE_04 and HAVE_NN_FOLDS):
        raise RuntimeError("To train 05 you need: features + lgbm + pyboost + NN fold checkpoints (checkpoints_nn/fold_*.npz).")
    print("[RUN] 05_train_lgbm_meta.py")
    run_logged_with_eta(
        "05_train_lgbm_meta.py",
        typical_sec=4*3600,
        progress_re=r"target\s+(\d+)\s*/\s*(\d+)\s+done",
        interval_sec=ETA_INTERVAL_SEC
    )
    HAVE_05 = exists(DST["lgbm_meta"])
else:
    print("[SKIP] 05 (using checkpoint)")

# ---------------------------
# 5) Run 06 and 07 (these do NOT require NN folds after patch; they can use blend_data fallback)
# ---------------------------
if not exists(SUB_DIR / "blend.parquet"):
    print("[RUN] 06_blend.py")
    run_logged_with_eta("06_blend.py", typical_sec=20*60, interval_sec=ETA_INTERVAL_SEC)
else:
    print("[SKIP] 06")

if not exists(SUB_DIR / "stacking.parquet"):
    print("[RUN] 07_stacking.py")
    # 07 has its own detailed prints; ETA here is just rough
    run_logged_with_eta("07_stacking.py", typical_sec=15*60, interval_sec=ETA_INTERVAL_SEC)
else:
    print("[SKIP] 07")

print("\nDONE. Outputs:")
print("  submissions/blend.parquet:", exists(SUB_DIR / "blend.parquet"))
print("  submissions/stacking.parquet:", exists(SUB_DIR / "stacking.parquet"))
print("Logs:", LOG_DIR)

[PATCH] 06_blend.py patched to use blend_data.npz when NN folds missing
[RESTORE] {'blend': True, 'lgbm': True, 'lgbm_meta': True, 'pyboost': True}
[STATE] features=False | nn_sub=True | nn_folds=False | lgbm=True | pyboost=True | meta=True | blend=False | stacking=False
[RUN] 01_feature_engineering.py
[01_feature_engineering.py] elapsed 02:00 | ETA ~ 07:59 | last: [2/8] Null Pattern PCA...
[01_feature_engineering.py] elapsed 04:00 | ETA ~ 05:59 | last: m3 = np.nanmean(diff ** 3, axis=1)
[01_feature_engineering.py] elapsed 06:00 | ETA ~ 03:59 | last: [8/8] Saving features...
[01_feature_engineering.py] elapsed 08:00 | ETA ~ 01:59 | last: Train: (750000, 2261), Test: (250000, 2261)
[01_feature_engineering.py] DONE in 08:00 (log: 01_feature_engineering.py.log)
[SKIP] 02/04
[SKIP] 03 (using checkpoint)
[SKIP] 05 (using checkpoint)
[RUN] 06_blend.py
[06_blend.py] elapsed 02:00 | ETA ~ 17:59 | last: [2/3] Optimizing per-target weights...
[06_blend.py] elapsed 04:00 | ETA ~ 15:59 | last: [2/

In [4]:
# 3) Superblend (with progress + ETA): base(OoF-optimized) + stacking + blend
import time
import numpy as np
import polars as pl
from pathlib import Path
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata

DATA_DIR = "/kaggle/input/datasets/hatab123/data-fusion-contest-2026/"
ROOT = Path("/kaggle/working/data_fusion_track_2")

# -----------------
# sanity checks
# -----------------
req = [
    ROOT / "blend_artifacts" / "blend_data.npz",
    ROOT / "submissions" / "blend.parquet",
    ROOT / "submissions" / "stacking.parquet",
    Path(DATA_DIR) / "train_target.parquet",
    Path(DATA_DIR) / "sample_submit.parquet",
]
missing = [str(p) for p in req if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))

print("[INFO] loading targets...")
train_tgt = pl.read_parquet(f"{DATA_DIR}train_target.parquet")
target_cols = [c for c in train_tgt.columns if c.startswith("target_")]
predict_cols = [c.replace("target_", "predict_") for c in target_cols]

y = train_tgt.select(target_cols).to_numpy().astype(np.float32)
n_targets = len(target_cols)
n_train = y.shape[0]
print(f"[INFO] targets: {n_targets}, train rows: {n_train}")

print("[INFO] loading blend_data.npz...")
d = np.load(ROOT / "blend_artifacts" / "blend_data.npz")
for k in ["oof_nn","test_nn","oof_lgbm","test_lgbm","oof_pb","test_pb"]:
    if k not in d:
        raise KeyError(f"blend_data.npz missing key: {k} (has {list(d.keys())})")

oof_nn, test_nn = d["oof_nn"], d["test_nn"]
oof_lgbm, test_lgbm = d["oof_lgbm"], d["test_lgbm"]
oof_pb, test_pb = d["oof_pb"], d["test_pb"]

# shape checks
def check_shape(name, a, rows):
    if a.ndim != 2 or a.shape[1] != n_targets or a.shape[0] != rows:
        raise ValueError(f"{name} shape {a.shape} expected ({rows}, {n_targets})")

check_shape("oof_nn", oof_nn, n_train)
check_shape("oof_lgbm", oof_lgbm, n_train)
check_shape("oof_pb", oof_pb, n_train)

n_test = test_nn.shape[0]
check_shape("test_nn", test_nn, n_test)
check_shape("test_lgbm", test_lgbm, n_test)
check_shape("test_pb", test_pb, n_test)

print("[INFO] loading blend/stack submissions...")
sub_blend = pl.read_parquet(ROOT / "submissions" / "blend.parquet")
sub_stack = pl.read_parquet(ROOT / "submissions" / "stacking.parquet")

# ensure columns exist
for c in predict_cols:
    if c not in sub_blend.columns:
        raise KeyError(f"blend.parquet missing col: {c}")
    if c not in sub_stack.columns:
        raise KeyError(f"stacking.parquet missing col: {c}")

test_blend = sub_blend.select(predict_cols).to_numpy()
test_stack = sub_stack.select(predict_cols).to_numpy()

if test_blend.shape != (n_test, n_targets) or test_stack.shape != (n_test, n_targets):
    raise ValueError(f"submission shapes mismatch: blend={test_blend.shape}, stack={test_stack.shape}, expected {(n_test,n_targets)}")

# -----------------
# utils
# -----------------
def to_ranks(a: np.ndarray) -> np.ndarray:
    # ranks per column (float64)
    return np.column_stack([rankdata(a[:, i]) for i in range(a.shape[1])]).astype(np.float64)

def safe_auc(y_true, y_pred) -> float:
    # if degenerate target
    if np.unique(y_true).size < 2:
        return 0.5
    return float(roc_auc_score(y_true, y_pred))

def fmt(sec: float) -> str:
    sec = max(0, int(sec))
    h = sec // 3600
    m = (sec % 3600) // 60
    s = sec % 60
    return f"{h:02d}:{m:02d}:{s:02d}" if h else f"{m:02d}:{s:02d}"

# -----------------
# base blend: per-target weight search on OOF (rank space)
# -----------------
print("[INFO] ranking OOF/test preds...")
t0 = time.time()

r_nn, r_lgbm, r_pb = to_ranks(oof_nn), to_ranks(oof_lgbm), to_ranks(oof_pb)
rt_nn, rt_lgbm, rt_pb = to_ranks(test_nn), to_ranks(test_lgbm), to_ranks(test_pb)

oof_base = np.zeros_like(r_nn)
test_base = np.zeros_like(rt_nn)

grid = np.arange(0.0, 1.0001, 0.05)

print("[INFO] optimizing per-target weights on OOF (this can take a while)...")
best_ws = np.zeros((n_targets, 3), dtype=np.float64)
best_aucs = np.zeros((n_targets,), dtype=np.float64)

for j in range(n_targets):
    tj = time.time()
    yj = y[:, j]
    best_auc = -1.0
    best_w = (1/3, 1/3, 1/3)

    # brute grid over simplex
    for w0 in grid:
        for w1 in grid:
            if w0 + w1 > 1.0:
                continue
            w2 = 1.0 - w0 - w1
            pred = w0 * r_nn[:, j] + w1 * r_lgbm[:, j] + w2 * r_pb[:, j]
            auc = safe_auc(yj, pred)
            if auc > best_auc:
                best_auc = auc
                best_w = (w0, w1, w2)

    w0, w1, w2 = best_w
    best_ws[j] = (w0, w1, w2)
    best_aucs[j] = best_auc

    oof_base[:, j] = w0 * r_nn[:, j] + w1 * r_lgbm[:, j] + w2 * r_pb[:, j]
    test_base[:, j] = w0 * rt_nn[:, j] + w1 * rt_lgbm[:, j] + w2 * rt_pb[:, j]

    done = j + 1
    elapsed = time.time() - t0
    eta = (elapsed / done) * (n_targets - done) if done else 0
    if done <= 3 or done % 5 == 0 or done == n_targets:
        print(f"  target {done:02d}/{n_targets} | best_auc={best_auc:.5f} | w={best_w} | elapsed={fmt(elapsed)} | ETA~{fmt(eta)}")

print(f"[INFO] base optimization finished in {fmt(time.time()-t0)}")
print(f"[INFO] mean best AUC across targets (not macro!): {best_aucs.mean():.5f}")

# -----------------
# combine with existing blend/stack submissions (rank space)
# -----------------
print("[INFO] ranking blend/stack submissions...")
r_test_blend = to_ranks(test_blend)
r_test_stack = to_ranks(test_stack)

# final weights (you can tune)
w_base, w_stack, w_blend = 0.30, 0.55, 0.15
test_super = w_base * test_base + w_stack * r_test_stack + w_blend * r_test_blend

# -----------------
# save submission
# -----------------
sample = pl.read_parquet(f"{DATA_DIR}sample_submit.parquet")
sub = pl.DataFrame({"customer_id": sample["customer_id"]}).hstack(
    pl.DataFrame(test_super.astype(np.float64), schema=predict_cols)
)

out_dir = ROOT / "submissions"
out_dir.mkdir(exist_ok=True)
out_path = out_dir / "superblend.parquet"
sub.write_parquet(out_path)

print("[DONE] Saved:", out_path)
print("[DONE] shape:", sub.shape)

[INFO] loading targets...
[INFO] targets: 41, train rows: 750000
[INFO] loading blend_data.npz...
[INFO] loading blend/stack submissions...
[INFO] ranking OOF/test preds...
[INFO] optimizing per-target weights on OOF (this can take a while)...
  target 01/41 | best_auc=0.92227 | w=(np.float64(0.30000000000000004), np.float64(0.45), np.float64(0.24999999999999994)) | elapsed=01:00 | ETA~40:12
  target 02/41 | best_auc=0.84562 | w=(np.float64(0.4), np.float64(0.25), np.float64(0.35)) | elapsed=01:40 | ETA~32:38
  target 03/41 | best_auc=0.88080 | w=(np.float64(0.35000000000000003), np.float64(0.4), np.float64(0.2499999999999999)) | elapsed=02:18 | ETA~29:13
  target 05/41 | best_auc=0.91116 | w=(np.float64(0.25), np.float64(0.4), np.float64(0.35)) | elapsed=03:29 | ETA~25:06
  target 10/41 | best_auc=0.80497 | w=(np.float64(0.4), np.float64(0.2), np.float64(0.39999999999999997)) | elapsed=06:22 | ETA~19:45
  target 15/41 | best_auc=0.91587 | w=(np.float64(0.15000000000000002), np.float64

In [7]:
import numpy as np
import polars as pl
from pathlib import Path

ROOT = Path("/kaggle/working/data_fusion_track_2/submissions")
stack = pl.read_parquet(ROOT/"stacking.parquet")
meta  = pl.read_parquet(ROOT/"lgbm_meta.parquet")

pred_cols = [c for c in stack.columns if c.startswith("predict_")]
cid = stack["customer_id"]

A = stack.select(pred_cols).to_numpy()
B = meta.select(pred_cols).to_numpy()

def ranks(a):
    n, m = a.shape
    r = np.empty((n, m), dtype=np.float32)
    for j in range(m):
        idx = np.argsort(a[:, j], kind="mergesort")
        r[idx, j] = np.arange(1, n+1, dtype=np.float32)
    return r

rA = ranks(A)
rB = ranks(B)

w_stack, w_meta = 0.95, 0.05
mix = w_stack * rA + w_meta * rB

out = pl.DataFrame({"customer_id": cid}).hstack(
    pl.DataFrame(mix.astype(np.float64), schema=pred_cols)
)
out_path = ROOT/"submit_one_shot_stack95_meta05.parquet"
out.write_parquet(out_path)
print("Saved:", out_path)

Saved: /kaggle/working/data_fusion_track_2/submissions/submit_one_shot_stack95_meta05.parquet


In [ ]:
while True:
    continue


KeyboardInterrupt

